In [58]:
import pandas as pd
import numpy as np
import sklearn

In [59]:
# load data
data = pd.read_csv("Advanced_Regression_HousePrice_Dataset_3800 - Advanced_Regression_HousePrice_Dataset_3800.csv.csv")

- Part B: Dataset Understanding & Preparation
- Tasks:

    6. Identify features and target variable.

    7. Perform a train-test split.
    
    8. Apply basic preprocessing (scaling where required).

- Part C: Regularized Linear Models

    9. Implement Ridge Regression (L2).

    10. Implement Lasso Regression (L1).

    11. Tune the regularization parameter (a) using cross-validation.

    12. Compare Ridge and Lasso based on:
        - Training error
        - Validation error
        - Feature coefficient behavior

In [60]:
# remove unnecessary columns
data = data.drop(columns={"property_id"})

In [61]:
# cap outliers
def cap_outliers(data):
    for i in data.columns[1:9]:
        q1 = np.quantile(data[i], 0.25)
        q3 = np.quantile(data[i], 0.75)
        iqr = q3 - q1
        lower_bound = q1 - (1.5 * iqr)
        upper_bound = q3 + (1.5 * iqr)
        data[i] = data[i].clip(lower_bound, upper_bound)
cap_outliers(data)

In [62]:
# create standard, min-max scaler
std_scaler = sklearn.preprocessing.StandardScaler()
min_max_scaler = sklearn.preprocessing.MinMaxScaler()

In [63]:
# combine preprocessing techniques
# preprocessor = sklearn.compose.ColumnTransformer([
#     ('std',std_scaler,["area_sqft","distance_city_km"]),
#     ('min-max',min_max_scaler,["property_age"])
# ])

In [64]:
# convert sale_date type into datetime
data["sale_date"] = pd.to_datetime(data["sale_date"])
data["sale_year"] = data["sale_date"].dt.year
data["sale_month"] = data["sale_date"].dt.month
data =data.drop(columns={"sale_date"})

In [65]:
# split the data and idenfity the input features and target variable
x = data.drop(columns={"house_price_inr"},axis=1)
y = data["house_price_inr"]

x_train, x_test, y_train, y_test = sklearn.model_selection.train_test_split(x, y, test_size=0.2, random_state=0)
print(f"Training set shape:{x_train.shape}")
print(f"Testing set shape:{x_test.shape}")

Training set shape:(3040, 11)
Testing set shape:(760, 11)


In [66]:
linear_models = {"Linear Regression":sklearn.linear_model.LinearRegression(),
                 "Lasso Regression":sklearn.linear_model.Lasso(alpha=100),
                 "Ridge Regression":sklearn.linear_model.Ridge(alpha=100)}

best_linear_r2_score = 0
best_linear_model = ""

for i in linear_models:
    # build a pipeline
    # scaling reduce the model performance, that's why we used original data
    linear_model_pipeline = sklearn.pipeline.Pipeline([
        # ('preprocess',preprocessor),
        ('algo',linear_models[i])
    ])

    # fit the model
    linear_model_pipeline.fit(x_train, y_train)

    # predict the target variable
    y_pred_using_linear = linear_model_pipeline.predict(x_test)

    # evaluate the model
    print(f"R2 Score using {i}: {sklearn.metrics.r2_score(y_test, y_pred_using_linear)}")
    
    print(f"Mean Absolute Error using {i}: {sklearn.metrics.mean_absolute_error(y_test, y_pred_using_linear)}")
    print(f"Root Mean Squared Error using {i}: {sklearn.metrics.root_mean_squared_error(y_test, y_pred_using_linear)}\n")

    # find the best linear model in term of r2 score
    if best_linear_r2_score < sklearn.metrics.r2_score(y_test, y_pred_using_linear):
        best_linear_r2_score = sklearn.metrics.r2_score(y_test, y_pred_using_linear)
        best_linear_model = i

R2 Score using Linear Regression: 0.9153105303065608
Mean Absolute Error using Linear Regression: 1948927.2484571175
Root Mean Squared Error using Linear Regression: 2537592.142628462

R2 Score using Lasso Regression: 0.9153116984301666
Mean Absolute Error using Lasso Regression: 1948904.6165204218
Root Mean Squared Error using Lasso Regression: 2537574.642037681

R2 Score using Ridge Regression: 0.9151800336705672
Mean Absolute Error using Ridge Regression: 1946342.6852451244
Root Mean Squared Error using Ridge Regression: 2539546.4574486166



- Part D: Cross-Validation Strategies

13. Apply and compare the following cross-validation techniques:
    
    - K-Fold Cross-Validation
    - Stratified K-Fold Cross-Validation (by binning target values)
    - Leave-One-Out Cross-Validation
    - Time Series Split (using a time-related feature)

14. Analyze how performance metrics vary across different CV strategies.

In [67]:
# Implement k-fold cross-validation on Linear Regression
k_folds = sklearn.model_selection.KFold(n_splits=5)

# evaluate the score
# cv parameter take the k-fold object
score = sklearn.model_selection.cross_val_score(sklearn.linear_model.LinearRegression(), x, y, cv = k_folds)

print(f"Cross Validation Score: {score}")
print(f"Cross Validation Score Mean: {score.mean()}")

Cross Validation Score: [0.92333023 0.91654797 0.91906055 0.90376047 0.9194664 ]
Cross Validation Score Mean: 0.9164331236622564


In [68]:
# Implement k-fold cross-validation on Linear Regression
st_k_folds = sklearn.model_selection.StratifiedKFold(n_splits=2)

# evaluate the score
# cv parameter take the k-fold object
score = sklearn.model_selection.cross_val_score(sklearn.linear_model.LinearRegression(), x, y, cv = st_k_folds)

print(f"Cross Validation Score: {score}")
print(f"Cross Validation Score Mean: {score.mean()}")

Cross Validation Score: [0.9128904 0.9195718]
Cross Validation Score Mean: 0.9162310975110648


c:\Python311\Lib\site-packages\sklearn\model_selection\_split.py:784: UserWarning: The number of unique classes is greater than 50% of the number of samples. `y` could represent a regression problem, not a classification problem.
  type_of_target_y = type_of_target(y)
c:\Python311\Lib\site-packages\sklearn\model_selection\_split.py:811: UserWarning: The least populated class in y has only 1 members, which is less than n_splits=2.
  warnings.warn(


In [69]:
# Implement k-fold cross-validation on Linear Regression
time_series_cv = sklearn.model_selection.TimeSeriesSplit(n_splits=5)

# evaluate the score
# cv parameter take the k-fold object
score = sklearn.model_selection.cross_val_score(sklearn.linear_model.LinearRegression(), x, y, cv = time_series_cv)

print(f"Cross Validation Score: {score}")
print(f"Cross Validation Score Mean: {score.mean()}")

Cross Validation Score: [0.91307904 0.91907816 0.91616503 0.90568156 0.91958604]
Cross Validation Score Mean: 0.9147179652654996


- Part E: Tree-Based Regression Models

15. Implement Decision Tree Regression.

16. Control tree complexity using hyperparameters (max depth, min samples).

17. Implement Random Forest Regression.

18. Compare single-tree vs ensemble performance.

In [70]:
ensemble_methods = {"decision-tree":sklearn.tree.DecisionTreeRegressor(max_depth=7),
                    "random-forest":sklearn.ensemble.RandomForestRegressor(max_depth=6, min_samples_split=3)}

best_ensemble_r2_score = 0
best_ensemble_model = ""

# build a pipeline
for i in ensemble_methods:
    ensemble_method_pipeline = sklearn.pipeline.Pipeline([
        ('algo',ensemble_methods[i])
    ])
    
    # fit the model
    ensemble_method_pipeline.fit(x_train, y_train)
    
    # predict the target
    y_pred_using_ensemble = ensemble_method_pipeline.predict(x_test)
    
    # evaluate the model
    print(f"R2 Score using {i}: {sklearn.metrics.r2_score(y_test, y_pred_using_ensemble)}")
    print(f"Mean Absolute Error using {i}: {sklearn.metrics.mean_absolute_error(y_test, y_pred_using_ensemble)}\n")
    
    # find the best ensemble model in term of r2 score
    if best_ensemble_r2_score < sklearn.metrics.r2_score(y_test, y_pred_using_ensemble):
        best_ensemble_r2_score = sklearn.metrics.r2_score(y_test, y_pred_using_ensemble)
        best_ensemble_model = i

R2 Score using decision-tree: 0.8996693755756451
Mean Absolute Error using decision-tree: 2094663.3339618717

R2 Score using random-forest: 0.9219012844708132
Mean Absolute Error using random-forest: 1860867.4288015817



- Part F: Support Vector Regression

19. Implement Support Vector Regression (SVR) with:
- Linear kernel
- Polynomial or RBF kernel

20. Tune hyperparameters (C, y, ε).

21. Compare SVR performance with linear and tree-based models.

In [71]:
# build svr pipeline
svr_pipeline = sklearn.pipeline.Pipeline([
    ('algo',sklearn.svm.SVR())
])
# fit the model
svr_pipeline.fit(x_train, y_train)

# predict the house price
y_pred_using_svr = svr_pipeline.predict(x_test)

# evaluate the model
print(f"R2 Score using SVR: {sklearn.metrics.r2_score(y_test, y_pred_using_svr)}")
print(f"Mean Absolute Error using SVR: {sklearn.metrics.mean_absolute_error(y_test, y_pred_using_svr)}")

svr_r2_score = sklearn.metrics.r2_score(y_test, y_pred_using_svr)
svr_model = "Support Vector Regressor"

R2 Score using SVR: -0.027745164849784176
Mean Absolute Error using SVR: 6841648.644080992


- Part G: Model Comparison & Evaluation

    22. Evaluate all models using appropriate regression metrics:
        - MSE, MAE, RMSE
        - R2 Score
    23. Compare:
        - Regularized Linear Models
        - Tree-Based Models
        - Support Vector Regression
    24. Identify signs of overfitting or underfitting in each model.

In [72]:
# do the model comparison in term of r2 score and find the best model for this dataset

if best_linear_r2_score > best_ensemble_r2_score > svr_r2_score:
    print(f"Best model for this dataset is {best_linear_model} with {best_linear_r2_score} R2 Score")
elif best_ensemble_r2_score > best_linear_r2_score > svr_r2_score:
    print(f"Best model for this dataset is {best_ensemble_model} with {best_ensemble_r2_score} R2 Score")
else:
    print(f"Best model for this dataset is {svr_model} with {svr_r2_score} R2 Score")

Best model for this dataset is random-forest with 0.9219012844708132 R2 Score


- Part H: Final Analysis & Reporting

    25. Prepare a final report covering:
        - Best-performing model and justification
        - Impact of regularization
        - Role of cross-validation in model stability
        - Comparison of linear vs non-linear regressors
        - Business interpretation of results
    26. Submit:
        - Source code / notebook
        - Evaluation tables and plots
        - Final conclusions